# Education Data — Cleaning & Feature Engineering
**Project:** Student Academic Health Analysis  
**Audience:** ENG. DONIA — Data Analysis Evaluation Committee  
**Stack:** Python → Power BI  
**Phase:** 2 of 3 (SQL Storytelling → **Python Cleaning** → Power BI Dashboard)

## 1. Setup

In [3]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

import warnings

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)

## 2. Load Data

In [4]:
path = 'data/raw/'

students = pd.read_csv(path + 'students.csv')
attendance = pd.read_csv(path + 'attendance.csv')
homework = pd.read_csv(path + 'homework.csv')
performance = pd.read_csv(path + 'performance.csv')
comm = pd.read_csv(path + 'teacher_parent_communication.csv')

print('students:', students.shape)
print('attendance:', attendance.shape)
print('homework:', homework.shape)
print('performance:', performance.shape)
print('comm:', comm.shape)

students: (12156, 5)
attendance: (364680, 4)
homework: (60780, 7)
performance: (36468, 5)
comm: (24312, 4)


## 3. Clean — students

In [5]:
students['Student_ID'] = students['Student_ID'].str.strip()

# Two date formats coexist: YYYY-MM-DD and MM-DD-YYYY
students['Date_of_Birth'] = pd.to_datetime(students['Date_of_Birth'], format='mixed', dayfirst=False)
students['Age'] = (pd.Timestamp.today() - students['Date_of_Birth']).dt.days // 365

# Ordinal column for grade sorting in Power BI
grade_map = {'Grade 1': 1, 'Grade 2': 2, 'Grade 3': 3, 'Grade 4': 4, 'Grade 5': 5}
students['Grade_Num'] = students['Grade_Level'].map(grade_map)

print('Unparsed DOB:', students['Date_of_Birth'].isna().sum())
students.head()

Unparsed DOB: 0


,Student_ID,Full_Name,Date_of_Birth,Grade_Level,Emergency_Contact,Age,Grade_Num
0,S00001,Donna Williams,2007-02-10,Grade 3,781-534-4258x9046,19,3
1,S00002,John Stafford,2014-11-26,Grade 5,+1-782-691-6291x99704,11,5
2,S00003,Chad Harper,2017-02-07,Grade 3,308.517.3750,9,3
3,S00004,Anthony Martin,2014-11-10,Grade 5,306-771-1524x116,11,5
4,S00005,Mary Stone,2016-04-01,Grade 3,+1-794-484-8495x7772,10,3


## 4. Clean — attendance

In [6]:
attendance['Student_ID'] = attendance['Student_ID'].str.strip()
attendance['Date'] = pd.to_datetime(attendance['Date'])

# Inconsistent casing, leading spaces, 'absnt' typo — all normalized here
status_map = {
    'PRESENT': 'Present',
    'ABSENT': 'Absent',
    'ABSNT': 'Absent',
    'LATE': 'Late',
    'LEFT EARLY': 'Left Early',
    'EXCUSED': 'Excused'
}

attendance['Attendance_Status'] = (
    attendance['Attendance_Status']
    .str.strip()
    .str.upper()
    .map(status_map)
)

attendance['Month'] = attendance['Date'].dt.to_period('M').astype(str)

print(attendance['Attendance_Status'].value_counts())
print('Nulls:', attendance['Attendance_Status'].isna().sum())

Attendance_Status
Present       91614
Absent        91229
Late          91017
Left Early    45435
Excused       45385
Name: count, dtype: int64
Nulls: 0


## 5. Clean — homework

In [7]:
homework['Student_ID'] = homework['Student_ID'].str.strip()
homework['Due_Date'] = pd.to_datetime(homework['Due_Date'], format='mixed')

# Emoji and text variants mapped to three clean labels
done_vals = {'✔', '✅', ' Done ', 'Done'}
not_done_vals = {'❌', 'not done'}
pending_vals = {'pending'}

def map_status(val):
    v = str(val).strip()
    if v in done_vals: return 'Done'
    if v in not_done_vals: return 'Not Done'
    if v in pending_vals: return 'Pending'
    return np.nan

homework['Status'] = homework['Status'].apply(map_status)

print(homework['Status'].value_counts())
print('Unmapped:', homework['Status'].isna().sum())

Status
Done        30381
Not Done    20385
Pending     10014
Name: count, dtype: int64
Unmapped: 0


In [8]:
# Blank string treated as missing, not a third category
homework['Guardian_Signature'] = (
    homework['Guardian_Signature']
    .str.strip()
    .replace({'Yes': True, 'No': False, '': np.nan})
)

# Guardian_Signature nulls (20,060 rows) filled with False.
# A missing signature means we have no evidence of parental sign-off,
# so treating it as unsigned is the conservative and correct assumption.
homework['Guardian_Signature'] = homework['Guardian_Signature'].fillna(False)

print(homework['Guardian_Signature'].value_counts())

Guardian_Signature
False    40432
True     20348
Name: count, dtype: int64


/tmp/ipykernel_19149/2314154568.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  homework['Guardian_Signature'] = homework['Guardian_Signature'].fillna(False)


## 6. Clean — performance

In [9]:
performance['Student_ID'] = performance['Student_ID'].str.strip()

# Exam_Score range is 40-110, all values are valid
print('Exam_Score range:', performance['Exam_Score'].min(), '-', performance['Exam_Score'].max())

Exam_Score range: 40 - 110


In [10]:
# Mixed '%' suffix + -5 invalid values
performance['Homework_Completion_%'] = (
    performance['Homework_Completion_%']
    .str.replace('%', '', regex=False)
    .astype(float)
    .where(lambda x: x >= 0, np.nan)
)

# Homework_Completion_% nulls (7,376 rows) filled with per-subject median.
# Completion rates vary by subject (Arabic median = 95, English = 90),
# so a global median would introduce bias. Subject-level median is more accurate.
performance['Homework_Completion_%'] = performance.groupby('Subject')['Homework_Completion_%'].transform(
    lambda x: x.fillna(x.median())
)

print('Remaining nulls:', performance['Homework_Completion_%'].isna().sum())
print(performance.groupby('Subject')['Homework_Completion_%'].median())

Remaining nulls: 0
Subject
Arabic       95.0
English      90.0
Geography    95.0
History      95.0
Math         95.0
Science      95.0
Name: Homework_Completion_%, dtype: float64


## 7. Clean — communication

In [11]:
comm['Student_ID'] = comm['Student_ID'].str.strip()
comm['Date'] = pd.to_datetime(comm['Date'])
comm['Month'] = comm['Date'].dt.to_period('M').astype(str)

comm.head()

,Student_ID,Date,Message_Type,Message_Content,Month
0,S01133,2025-01-20,Automated Reminder,Cut authority ball must cut to maintain think ...,2025-01
1,S11179,2024-11-16,Parent to Teacher,Far baby different conference evening gas floo...,2024-11
2,S09537,2024-12-03,Parent to Teacher,According this reality reality wish join seaso...,2024-12
3,S00478,2025-01-25,Teacher to Parent,Feel article ever success choose serve forget ...,2025-01
4,S04736,2024-10-15,Parent to Teacher,Friend wonder contain customer operation half ...,2024-10


## 8. Validation — referential integrity

In [12]:
all_ids = set(students['Student_ID'])

checks = {
    'attendance': attendance['Student_ID'],
    'homework': homework['Student_ID'],
    'performance': performance['Student_ID'],
    'comm': comm['Student_ID']
}

for table, ids in checks.items():
    orphans = set(ids.unique()) - all_ids
    print(f'{table} — orphan IDs: {len(orphans)}')

attendance — orphan IDs: 0
homework — orphan IDs: 0
performance — orphan IDs: 0
comm — orphan IDs: 0


## 9. Near-Duplicate Column Flags

In [13]:
# students:
# Date_of_Birth and Age carry the same information — Age is the derived form.
# Date_of_Birth is kept only for the export; Age is what Power BI uses.

# attendance:
# No near-duplicates detected.

# homework:
# No near-duplicates detected.
# Note: Guardian_Signature and Status are related but not duplicates —
# one tracks parental involvement, the other tracks submission outcome.

# performance:
# Homework_Completion_% here is per-exam-record level.
# HW_Completion_% in student_summary is the student-level average.
# Different granularities — not duplicates, but worth noting.

# comm:
# No near-duplicates detected.

print('Near-duplicate flag review complete — see comments above.')

Near-duplicate flag review complete — see comments above.


## 10. Feature Engineering

In [14]:
att_rate = (
    attendance.groupby('Student_ID')
    .apply(lambda x: (x['Attendance_Status'] == 'Present').sum() / len(x) * 100)
    .round(2)
    .reset_index()
    .rename(columns={0: 'Attendance_Rate_%'})
)

hw_rate = (
    homework.groupby('Student_ID')
    .apply(lambda x: (x['Status'] == 'Done').sum() / len(x) * 100)
    .round(2)
    .reset_index()
    .rename(columns={0: 'HW_Completion_%'})
)

avg_score = (
    performance.groupby('Student_ID')['Exam_Score']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'Exam_Score': 'Avg_Exam_Score'})
)

missed = (
    homework[homework['Status'] == 'Not Done']
    .groupby('Student_ID')
    .size()
    .reset_index()
    .rename(columns={0: 'Missed_Assignments'})
)

comm_count = (
    comm.groupby('Student_ID')
    .size()
    .reset_index()
    .rename(columns={0: 'Comm_Count'})
)

/tmp/ipykernel_19149/1992578892.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x['Attendance_Status'] == 'Present').sum() / len(x) * 100)
/tmp/ipykernel_19149/1992578892.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x['Status'] == 'Done').sum() / len(x) * 100)


In [15]:
student_summary = (
    students[['Student_ID', 'Full_Name', 'Grade_Level', 'Grade_Num', 'Age']]
    .merge(att_rate, on='Student_ID', how='left')
    .merge(hw_rate, on='Student_ID', how='left')
    .merge(avg_score, on='Student_ID', how='left')
    .merge(missed, on='Student_ID', how='left')
    .merge(comm_count, on='Student_ID', how='left')
)

student_summary['Missed_Assignments'] = student_summary['Missed_Assignments'].fillna(0).astype(int)
student_summary['Comm_Count'] = student_summary['Comm_Count'].fillna(0).astype(int)
student_summary['Has_Communication'] = student_summary['Comm_Count'] > 0

# Same criteria defined in SQL Chapter 6
student_summary['Is_At_Risk'] = (
    (student_summary['Attendance_Rate_%'] < 20) &
    (student_summary['Avg_Exam_Score'] < 60) &
    (student_summary['Missed_Assignments'] > 3)
)

def score_band(score):
    if pd.isna(score): return 'Unknown'
    if score < 60: return 'Failing'
    if score < 75: return 'Average'
    if score < 90: return 'Good'
    return 'Excellent'

student_summary['Score_Band'] = student_summary['Avg_Exam_Score'].apply(score_band)

print('At-risk:', student_summary['Is_At_Risk'].sum())
print(student_summary.shape)
student_summary.head()

At-risk: 37
(12156, 13)


,Student_ID,Full_Name,Grade_Level,Grade_Num,Age,Attendance_Rate_%,HW_Completion_%,Avg_Exam_Score,Missed_Assignments,Comm_Count,Has_Communication,Is_At_Risk,Score_Band
0,S00001,Donna Williams,Grade 3,3,19,33.33,33.33,NaN,1,3,True,False,Unknown
1,S00002,John Stafford,Grade 5,5,11,44.00,50.00,83.5,1,4,True,False,Good
2,S00003,Chad Harper,Grade 3,3,9,20.00,40.00,63.0,2,0,False,False,Average
3,S00004,Anthony Martin,Grade 5,5,11,16.67,60.00,82.5,2,3,True,False,Good
4,S00005,Mary Stone,Grade 3,3,10,27.59,75.00,70.5,1,0,False,False,Average


## 11. Drop Unused Columns

In [16]:
# students:
# Emergency_Contact — no dashboard page uses it
# Date_of_Birth — Age is the derived column used in Power BI
students.drop(columns=['Emergency_Contact', 'Date_of_Birth'], inplace=True)
print('students:', students.shape, '|', students.columns.tolist())

students: (12156, 5) | ['Student_ID', 'Full_Name', 'Grade_Level', 'Age', 'Grade_Num']


In [18]:
# homework:
# Assignment_Name — too granular for any dashboard visual, not used in any page
homework.drop(columns=['Assignment_Name'], inplace=True)
print('homework:', homework.shape, '|', homework.columns.tolist())

homework: (60780, 6) | ['Student_ID', 'Subject', 'Due_Date', 'Status', 'Grade_Feedback', 'Guardian_Signature']


In [19]:
# performance:
# Teacher_Comments — free text, no dashboard page uses it
performance.drop(columns=['Teacher_Comments'], inplace=True)
print('performance:', performance.shape, '|', performance.columns.tolist())

performance: (36468, 4) | ['Student_ID', 'Subject', 'Exam_Score', 'Homework_Completion_%']


In [20]:
# comm:
# Message_Content — free text, no dashboard page uses it
comm.drop(columns=['Message_Content'], inplace=True)
print('comm:', comm.shape, '|', comm.columns.tolist())

comm: (24312, 4) | ['Student_ID', 'Date', 'Message_Type', 'Month']


## 12. Final Checks

In [21]:
for name, df in [('students', students), ('attendance', attendance), ('homework', homework),
                 ('performance', performance), ('comm', comm), ('student_summary', student_summary)]:
    print(f'--- {name} ---')
    print('shape:', df.shape)
    print('nulls:\n', df.isna().sum()[df.isna().sum() > 0])
    print()

--- students ---
shape: (12156, 5)
nulls:
 Series([], dtype: int64)

--- attendance ---
shape: (364680, 5)
nulls:
 Series([], dtype: int64)

--- homework ---
shape: (60780, 6)
nulls:
 Series([], dtype: int64)

--- performance ---
shape: (36468, 4)
nulls:
 Series([], dtype: int64)

--- comm ---
shape: (24312, 4)
nulls:
 Series([], dtype: int64)

--- student_summary ---
shape: (12156, 13)
nulls:
 HW_Completion_%     91
Avg_Exam_Score     576
dtype: int64



## 13. Export

In [22]:
out = 'data/clean/'
os.makedirs(out, exist_ok=True)

student_summary.to_csv(out + 'student_summary.csv', index=False)
attendance.to_csv(out + 'attendance.csv', index=False)
homework.to_csv(out + 'homework.csv', index=False)
performance.to_csv(out + 'performance.csv', index=False)
comm.to_csv(out + 'communication.csv', index=False)

print('Exported to', out)

Exported to data/clean/
